ПАЙПЛАЙН: слово → значение → семантический класс


что делаем:
- грузим датасет с определениями (слово → значение)
- переводим определения в векторы
- сопоставляем

1. потом подаем в код слово в контексте
например: "солдат держал в руках автомат"

2. энкодим

3. ищем ближайшую дефиницию (по косинусной близости)
→ определение "ручное автоматическое скорострельное оружие"

4. подбираем тег(и)
→ семантические теги "instr" и "tool" и "weapon"




---



# Грузим датасет с дефинициями



In [ ]:
!pip install torch transformers scikit-learn sentence-transformers


In [ ]:
# В Colab исходные данные и модели читаются с Google Drive.
# Результаты, Trackio и checkpoints сохраняются локально внутри клонированного git-проекта.
import importlib.util
import os
from pathlib import Path


IS_COLAB = (importlib.util.find_spec('google') is not None and
             importlib.util.find_spec('google.colab') is not None)


def find_project_root(start=None):
    """Находит корень локально клонированного git-репозитория."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError(
        'Корень git-репозитория не найден. В Colab сначала клонируйте проект в /content '
        'и перейдите в его папку командой %cd.'
    )


PROJECT_ROOT = find_project_root()
if IS_COLAB:
    from google.colab import drive
    # Google Drive используется только как постоянное хранилище входных данных и моделей.
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Аспирантура')
else:
    # Вне Colab структура data/models берётся из локального git-проекта.
    DRIVE_ROOT = PROJECT_ROOT

# Поддерживаются обе структуры: DRIVE_ROOT/data и старый вариант DRIVE_ROOT/conllu.
DATA_ROOT = DRIVE_ROOT / 'data' if (DRIVE_ROOT / 'data').exists() else DRIVE_ROOT
MODELS_ROOT = DRIVE_ROOT / 'models' if (DRIVE_ROOT / 'models').exists() else DRIVE_ROOT

# Все изменяемые и промежуточные файлы остаются на локальном диске runtime.
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
CACHE_ROOT = OUTPUT_ROOT / 'experiment_cache'
LOCAL_CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints'
TRACKIO_ROOT = OUTPUT_ROOT / 'trackio'
for directory in (OUTPUT_ROOT, CACHE_ROOT, LOCAL_CHECKPOINT_ROOT, TRACKIO_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ['TRACKIO_DIR'] = str(TRACKIO_ROOT)

print('Режим:', 'Colab + Google Drive' if IS_COLAB else 'локальный запуск')
print('Git-проект и checkpoints:', PROJECT_ROOT)
print('Данные:', DATA_ROOT)
print('Модели:', MODELS_ROOT)
print('Checkpoints:', LOCAL_CHECKPOINT_ROOT)

# Кэш загружаемых датасетов и MiniLM хранится на Google Drive вместе с входными ресурсами.
MODEL_CACHE = MODELS_ROOT / 'huggingface'
DATASET_CACHE = DATA_ROOT / 'dictionary_dataset'
MODEL_CACHE.mkdir(parents=True, exist_ok=True)


In [ ]:
from datasets import load_dataset, load_from_disk

if (DATASET_CACHE / 'dataset_dict.json').exists():
    ds = load_from_disk(str(DATASET_CACHE))
else:
    ds = load_dataset(
        'snagbreac/russian-reverse-dictionary-train-data',
        cache_dir=str(DATA_ROOT / 'huggingface_cache'),
    )
    ds.save_to_disk(str(DATASET_CACHE))


In [ ]:
print(len(ds['train']))


In [ ]:
print(ds['train'][1000])


In [ ]:
from collections import defaultdict

# индексируем определения чтобы удобнее обращаться
#TODO выгрузить чтобы не надо было каждый раз индексировать
word2defs = defaultdict(list)

for entry in ds['train']:
    word2defs[entry['word']].append(entry['definition'])


In [ ]:
automat_definitions = word2defs.get('автомат')
print(automat_definitions)


In [ ]:
#TODO: почистить от помет (напр: разг. дает тег "speech" к дефиниции, НО при этом понижает точность наиболее вероятных тегов)


# Грузим эмбеддинги для перевода контекстов, дефиниций и тегов в вектор


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

LOCAL_MODEL = MODELS_ROOT / 'paraphrase-multilingual-MiniLM-L12-v2'
MODEL_NAME = str(LOCAL_MODEL) if (LOCAL_MODEL / 'config.json').exists() else 'paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(MODEL_NAME, cache_folder=str(MODEL_CACHE))


# Теги → эмбеддинги


In [ ]:
TAGS = {
    "abstr": "абстрактные понятия, не имеющие физической формы",
    "age": "возраст и возрастные характеристики",
    "animal": "животные и живые существа животного мира",
    "appear": "появление, возникновение чего-либо",
    "artwork": "произведения искусства как объекты творчества",
    "art": "искусство как деятельность и культурная сфера",
    "be": "существование, бытие, факт наличия",
    "behav": "поведение и модели действий",
    "berry": "ягоды как биологический класс плодов",
    "changest": "изменение состояния или характеристик",
    "cin": "кинематограф, кино как индустрия и искусство",
    "class": "класс, категория или группа объектов",
    "cloth": "одежда и текстильные изделия",
    "color": "цветовые характеристики и восприятие цвета",
    "concr": "конкретные предметные объекты",
    "constr": "здания и строительные сооружения",
    "contact": "контактирование и взаимодействие через соприкосновение или связь",
    "degr": "степень, мера интенсивности или выраженности",
    "device": "высокотехнологичные устройства и автоматические системы",
    "dir": "направление движения или ориентации",
    "disappear": "исчезновение или прекращение существования",
    "disease": "болезни и патологические состояния",
    "dish": "посуда и предметы кухонного использования",
    "dist": "расстояние между объектами",
    "doc": "документы и текстовые официальные материалы",
    "dram": "театр и драматическое искусство",
    "emot": "эмоции и эмоциональные состояния",
    "etn": "этнонимы и названия народов",
    "ev": "оценка, результат оценивания",
    "famn": "фамилии как имена семейного происхождения",
    "food": "еда и пищевые продукты",
    "form": "форма и структурные характеристики",
    "fruit": "плоды растений",
    "furn": "мебель и предметы обстановки",
    "grain": "зерновые культуры и зерно",
    "group": "группы людей или объединения",
    "hum": "человек как биологический и социальный субъект",
    "humq": "качества человека и личностные характеристики",
    "impact": "воздействие одного объекта на другой",
    "instr": "инструменты как средства выполнения действий",
    "inter": "взаимодействие между объектами или субъектами",
    "kin": "родственные отношения",
    "light": "свет и световые явления",
    "max": "большая степень, максимум значения",
    "ment": "ментальная и когнитивная сфера",
    "min": "малое значение, минимум",
    "move": "передвижение и движение",
    "mus": "музыка как искусство и звуковая система",
    "neg": "негативная оценка или отрицательная характеристика",
    "org": "организации и институциональные структуры",
    "part": "часть целого",
    "patrn": "отчество как часть имени",
    "perc": "восприятие и перцептивные процессы",
    "physiol": "физиологические процессы живых организмов",
    "physq": "физические свойства объектов",
    "place": "место и пространственные объекты",
    "plant": "растения и ботанические объекты",
    "posit": "позитивная оценка или положительная характеристика",
    "poss": "посессивная сфера, принадлежность",
    "prof": "профессии и виды профессиональной деятельности",
    "propn": "имена собственные",
    "psych": "психика и психические процессы",
    "put": "помещения и локализованные пространства",
    "qtm": "кванты и дискретные единицы",
    "quant": "количество и количественные характеристики",
    "set": "множества и совокупности объектов",
    "size": "размер и пространственные характеристики",
    "sound": "звук и акустические явления",
    "speech": "речь и речевая деятельность",
    "speed": "скорость движения или изменения",
    "stuff": "вещества и материалы",
    "taste": "вкус как сенсорная характеристика",
    "temper": "температура и тепловые характеристики",
    "text": "текстовые объекты и письменные данные",
    "time": "время и временные характеристики",
    "tool": "инструменты и орудия труда",
    "topon": "топонимы и географические названия",
    "transp": "транспорт и средства передвижения",
    "unit": "единицы измерения",
    "weapon": "оружие и военные средства",
    "weather": "погодные явления",
    "weight": "вес и масса объектов"
}


In [ ]:
tag_names = list(TAGS.keys())
tag_texts = list(TAGS.values())

tag_embeddings = model.encode(tag_texts)


In [ ]:
# близости тегов и что weapon близко к tool и instr

from sklearn.metrics.pairwise import cosine_similarity

def tag_similarity(tag1, tag2):
    i = tag_names.index(tag1)
    j = tag_names.index(tag2)

    sim = cosine_similarity(
        [tag_embeddings[i]],
        [tag_embeddings[j]]
    )[0][0]

    return float(sim)


In [ ]:
print("weapon ~ tool:", tag_similarity("weapon", "tool"))
print("weapon ~ instr:", tag_similarity("weapon", "instr"))
print("tool ~ instr:", tag_similarity("tool", "instr"))




---



In [ ]:
# а если брать дефиниции из словаря а не короткие описания...

weapon = word2defs.get('оружие')
# в описании тега: оружие и военные средства
for i in weapon:
  print(i)

print('///')

instr_or_tool = word2defs.get('инструмент')
# в описании тега: инструменты и орудия труда // инструменты как средства выполнения действий
for j in instr_or_tool:
  print(j)


# Дефиниции → эмбеддинги + подбираем теги


In [ ]:
# получаем список тегов для дефиниции

def assign_tags(definition, threshold=0.4, top_k=5):

    def_emb = model.encode([definition])

    sims = cosine_similarity(def_emb, tag_embeddings)[0]

    results = []

    # отбираем теги по порогу
    for tag, score in zip(tag_names, sims):
        if score >= threshold:
            results.append((tag, float(score)))

    # сортировка по релевантности
    results.sort(key=lambda x: x[1], reverse=True)

    if top_k:
        results = results[:top_k]

    return results


In [ ]:
# TODO фильтровать по словарям? тут 3 вида
for d in automat_definitions:
    print("\nDEF:", d)
    print("TAGS:", assign_tags(d))


In [ ]:
def1 = "разг. телефон-автомат"
print(assign_tags(def1, top_k=10))

def2 = "телефон-автомат"
print(assign_tags(def2, top_k=10))


# Парсим конлю


In [ ]:
!pip install conllu


In [ ]:
import re
SEM_PATTERN = re.compile(r"sem=\[([^\]]+)\]")

def extract_sem_tags(misc):
    """
    Универсальный извлекатель sem-тегов из MISC
    поддерживает dict и строку
    """
    if misc is None:
        return []

    # случай: conllu вернул dict
    if isinstance(misc, dict):
        sem_val = misc.get("sem")
        if not sem_val:
            return []
        sem_val = sem_val.strip("[]")
        return [t.strip() for t in sem_val.split(",")]

    # случай: строка
    match = SEM_PATTERN.search(str(misc))
    if not match:
        return []

    return [t.strip() for t in match.group(1).split(",")]


In [ ]:
from conllu import parse

def parse_conllu(text):
    sentences = parse(text)
    parsed = []

    for sent in sentences:
        tokens = []
        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))

            tokens.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "feats": token["feats"],
                "head": token["head"],
                "deprel": token["deprel"],
                "sem": sem_tags
            })

        parsed.append(tokens)

    return parsed


In [ ]:
text = '''
# sent_id = GramEval2020-test-fiction-2
# genre = fiction
# text = Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.
1	Может	мочь	VERB	_	Aspect=Imp|Mood=Ind|Number=Sing|Person=3|Tense=Pres|Transit=Intr|VerbForm=Fin|Voice=Act	4	parataxis:discourse	_	sem=[abstr]|hls
2	быть	быть	VERB	_	Transit=Intr|VerbForm=Inf|Voice=Act	1	xcomp	_	SpaceAfter=No
3	,	,	PUNCT	_	_	1	punct	_	_
4	потому	потому	ADVPRO	_	Degree=Pos|PronType=Dem	0	root	_	SpaceAfter=No
5	,	,	PUNCT	_	_	9	punct	_	_
6	что	что	SCONJ	_	_	9	mark	_	_
7	музыкальный	музыкальный	ADJ	_	Case=Nom|Degree=Pos|Gender=Masc|Number=Sing	8	amod	_	sem=[mus,sound]|hls
8	автомат	автомат	NOUN	_	Animacy=Inan|Case=Nom|Gender=Masc|Number=Sing	9	nsubj	_	sem=[concr,device,tool]|hls
9	грохотал	грохотать	VERB	_	Aspect=Imp|Gender=Masc|Mood=Ind|Number=Sing|Tense=Past|Transit=Intr|VerbForm=Fin|Voice=Act	4	advcl	_	sem=[sound]|hls
10	здесь	здесь	ADVPRO	_	Degree=Pos|PronType=Dem	9	advmod	_	sem=[place]|hls
11	не	не	PART	_	Polarity=Neg	12	advmod	_	_
12	так	так	ADVPRO	_	Degree=Pos|PronType=Dem	13	advmod	_	sem=[degr]|hls
13	оглушительно	оглушительно	ADV	_	Degree=Pos	9	advmod	_	SpaceAfter=No|sem=[sound]|hls
14	,	,	PUNCT	_	_	16	punct	_	_
15	как	как	SCONJ	_	_	16	mark	_	_
16	везде	везде	ADVPRO	_	Degree=Pos|PronType=Tot	12	advcl	_	SpaceAfter=No|sem=[place]|hls
17	.	.	PUNCT	_	_	4	punct	_	_
'''

parsed_text = parse_conllu(text)[0]
for token in parsed_text:
  print(token)
  print(token['sem'])


### Собираем словоупотребоения по группам тегов


In [ ]:
#TODO: доделать

def extract_examples(conllu_text, target_tags=None):
    sentences = parse(conllu_text)
    results = []

    if target_tags:
        target_tags = [t.strip().lower() for t in target_tags]

    for sent in sentences:
        sentence_text = sent.metadata.get("text", "")

        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))
            sem_tags = [t.lower() for t in sem_tags]

            if not sem_tags:
                continue

            if target_tags:
                if not any(tag in sem_tags for tag in target_tags):
                    continue

            results.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "sem": sem_tags,
                "sentence": sentence_text,
                "token_id": token["id"]
            })

    return results


In [ ]:
target_tags = ["abstr", "mus", "sound", "concr", "device", "tool", "degr", "place"]

finding_tags = extract_examples(text, target_tags)
for found_tags in finding_tags:
  print(found_tags)




---



к этому моменту у нас есть:

* готовый эмбеддинг (взяли paraphrase-multilingual-MiniLM-L12-v2) - заменить?
*   словарик с тегами
*   векторизованные теги
* датасет с дефинициями (взяли russian-reverse-dictionary-train-data)
* векторизованные дефиниции
* код для подбора наиболее близких тегов к дефинициям

что надо дальше: по контексту слова подбираем, какая из дефиниций больше всего подходит, и потом на основе этой дефиниции к слову в контексте присваивать теги

TODO: посмотреть семинар про ADAGRAM


#
